In [1]:
!pip freeze | grep scikit-learn

scikit-learn==1.6.1


In [2]:
!python -V

Python 3.9.6


In [1]:
import pickle
import pandas as pd

In [2]:
taxi_type = 'yellow'
year = 2023
month = 4
output_file = f'output/{taxi_type}-{year:04d}-{month:02d}.parquet'

In [3]:
with open('lin_reg.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [4]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [5]:
df = read_data(f'https://d37ci6vzurychx.cloudfront.net/trip-data/{taxi_type}_tripdata_{year:04d}-{month:02d}.parquet')

In [6]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [7]:
y_pred.std()

np.float64(6.353975123467482)

In [8]:
df_result = pd.DataFrame()
df_result['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')
df_result['predicted_duration'] = y_pred
df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [9]:
df_result

,ride_id,predicted_duration
0,2023/04_0,16.132230
1,2023/04_1,32.264742
2,2023/04_2,12.255534
3,2023/04_3,12.133333
4,2023/04_4,13.071936
...,...,...
3199710,2023/04_3288245,12.813459
3199711,2023/04_3288246,11.921462
3199712,2023/04_3288247,12.301362
3199713,2023/04_3288248,12.555935


In [ ]:
#jupyter nbconvert --to script h4_answers.ipynb